# 🧠 LSTM — Predicción del número semanal de tweets

Este notebook entrena un modelo **LSTM apilado** para predecir cuántos tweets publicará Elon Musk la semana siguiente, dado un historial de `SEQ_LEN` semanas previas.

**Estructura:**
1. Setup y carga de datos
2. Preprocesamiento y construcción de ventanas
3. Train / Val / Test split
4. Carga del modelo desde `models/lstm_model.py`
5. Entrenamiento con curvas de convergencia
6. Evaluación en train, validación y test
7. Resumen de resultados


## 1. Setup

In [ ]:
# ── Clonar repo y situarse en él ──────────────────────────────────────────────
import os

if not os.path.exists('AP'):
    !git clone https://github.com/0xnito/AP.git
%cd AP
!ls

In [ ]:
# ── Instalar dependencias ─────────────────────────────────────────────────────
!pip install -q kaggle tensorflow matplotlib scikit-learn

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

# ── Semilla de reproducibilidad ───────────────────────────────────────────────
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## 2. Carga de datos

In [ ]:
# ── Descargar dataset (Kaggle) o usar copia local ────────────────────────────
import os

os.makedirs('data/raw', exist_ok=True)
csv_path = 'data/raw/all_musk_posts.csv'
kaggle_json = '/root/.kaggle/kaggle.json'

if os.path.exists(csv_path):
    print('✅ Dataset ya disponible.')
else:
    if os.path.exists(kaggle_json):
        print('🔽 Descargando desde Kaggle...')
        !kaggle datasets download -d dadalyndell/elon-musk-tweets-2010-to-2025-march -p data/raw
        !unzip -o data/raw/elon-musk-tweets-2010-to-2025-march.zip -d data/raw
        print('✅ Descarga completada.')
    else:
        print(
            '⚠️  Sube kaggle.json o descarga manualmente:\n'
            'https://www.kaggle.com/datasets/dadalyndell/elon-musk-tweets-2010-to-2025-march\n'
            'y coloca all_musk_posts.csv en data/raw/'
        )

In [ ]:
# ── Cargar y preparar serie temporal semanal ──────────────────────────────────
df = pd.read_csv(csv_path)
df['createdAt'] = pd.to_datetime(df['createdAt'], utc=True)
df['createdAt_naive'] = df['createdAt'].dt.tz_localize(None)
df['week'] = df['createdAt_naive'].dt.to_period('W').apply(lambda r: r.start_time)

weekly = (
    df.groupby('week').size()
      .reset_index(name='tweet_count')
      .sort_values('week')
      .reset_index(drop=True)
)

print(f'Semanas totales: {len(weekly)}')
print(f'Rango: {weekly.week.min().date()} → {weekly.week.max().date()}')
weekly.describe()

In [ ]:
# ── Visualizar serie temporal ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(weekly['week'], weekly['tweet_count'], linewidth=0.8, alpha=0.9)
axes[0].set_title('Actividad semanal de tweets — Elon Musk (2010–2025)', fontsize=13)
axes[0].set_xlabel('Semana')
axes[0].set_ylabel('Nº tweets')

axes[1].plot(weekly['week'], weekly['tweet_count'].rolling(12).mean(),
             color='orange', linewidth=1.5, label='Media móvil 12 sem')
axes[1].fill_between(weekly['week'],
                     weekly['tweet_count'].rolling(12).mean() - weekly['tweet_count'].rolling(12).std(),
                     weekly['tweet_count'].rolling(12).mean() + weekly['tweet_count'].rolling(12).std(),
                     alpha=0.2, color='orange')
axes[1].set_title('Media móvil 12 semanas ± 1 std', fontsize=13)
axes[1].set_xlabel('Semana')
axes[1].set_ylabel('Nº tweets')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/raw/weekly_series.png', dpi=150)
plt.show()

## 3. Preprocesamiento

In [ ]:
# ── Hiperparámetros ───────────────────────────────────────────────────────────
SEQ_LEN     = 8    # ventana de entrada (semanas)
BATCH_SIZE  = 32
EPOCHS      = 150
PATIENCE    = 20   # early stopping

TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.15
# Test = 1 - TRAIN_FRAC - VAL_FRAC = 0.15

# ── Escalado Min-Max ──────────────────────────────────────────────────────────
values = weekly['tweet_count'].values.astype(np.float32).reshape(-1, 1)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(values).flatten()

# ── Construcción de ventanas deslizantes ──────────────────────────────────────
def make_windows(series, seq_len):
    X, y = [], []
    for i in range(len(series) - seq_len):
        X.append(series[i:i + seq_len])
        y.append(series[i + seq_len])
    return np.array(X)[..., np.newaxis], np.array(y)

X_all, y_all = make_windows(scaled, SEQ_LEN)

# ── Split cronológico (sin shuffle) ──────────────────────────────────────────
n = len(X_all)
n_train = int(n * TRAIN_FRAC)
n_val   = int(n * VAL_FRAC)

X_train, y_train = X_all[:n_train],          y_all[:n_train]
X_val,   y_val   = X_all[n_train:n_train+n_val], y_all[n_train:n_train+n_val]
X_test,  y_test  = X_all[n_train+n_val:],    y_all[n_train+n_val:]

print(f'Train: {len(X_train):>4}  |  Val: {len(X_val):>4}  |  Test: {len(X_test):>4}')
print(f'Shape de entrada: {X_train.shape}')

## 4. Modelo LSTM

In [ ]:
from models.lstm_model import build_lstm_model

model = build_lstm_model(
    seq_len=SEQ_LEN,
    n_features=1,
    lstm_units=64,
    dropout=0.2,
    learning_rate=1e-3,
)
model.summary()

total_params = model.count_params()
print(f'\n🔢 Parámetros totales: {total_params:,}')

## 5. Entrenamiento

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=10, min_lr=1e-6, verbose=1
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ── Curvas de entrenamiento ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Pérdida (MSE) — LSTM', fontsize=13)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'],     label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('MAE — LSTM', fontsize=13)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE (escalado)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Curvas de entrenamiento — Modelo LSTM', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('data/raw/lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Épocas entrenadas: {len(history.history["loss"])}')

## 6. Evaluación

In [ ]:
def evaluate_split(model, X, y_true_scaled, scaler, split_name):
    """Evalúa el modelo en un split y devuelve métricas en escala original."""
    y_pred_scaled = model.predict(X, verbose=0).flatten()

    # Invertir escalado
    y_true = scaler.inverse_transform(y_true_scaled.reshape(-1, 1)).flatten()
    y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100

    print(f'  {split_name:<12}  RMSE={rmse:7.2f}  MAE={mae:7.2f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return dict(split=split_name, RMSE=rmse, MAE=mae, R2=r2, MAPE=mape,
                y_true=y_true, y_pred=y_pred)

print('\n📊 Resultados LSTM (escala original, tweets/semana):')
print('─' * 65)
res_train = evaluate_split(model, X_train, y_train, scaler, 'Train')
res_val   = evaluate_split(model, X_val,   y_val,   scaler, 'Validación')
res_test  = evaluate_split(model, X_test,  y_test,  scaler, 'Test')
print('─' * 65)

In [ ]:
# ── Predicciones vs valores reales (Test) ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# — Test set
axes[0].plot(res_test['y_true'],  label='Real', linewidth=1.2)
axes[0].plot(res_test['y_pred'], label='Predicción LSTM', linewidth=1.2, alpha=0.85)
axes[0].set_title('Test — Real vs Predicción LSTM', fontsize=13)
axes[0].set_xlabel('Índice de semana (test)')
axes[0].set_ylabel('Nº tweets')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# — Scatter plot
axes[1].scatter(res_test['y_true'], res_test['y_pred'],
                alpha=0.6, edgecolors='k', linewidths=0.3, s=50)
lims = [min(res_test['y_true'].min(), res_test['y_pred'].min()),
        max(res_test['y_true'].max(), res_test['y_pred'].max())]
axes[1].plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[1].set_title(f'Scatter Real vs Predicho — Test (R²={res_test["R2"]:.4f})', fontsize=13)
axes[1].set_xlabel('Real')
axes[1].set_ylabel('Predicho')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/raw/lstm_predictions.png', dpi=150)
plt.show()

In [ ]:
# ── Tabla resumen final ───────────────────────────────────────────────────────
results_df = pd.DataFrame([
    {'Split': r['split'], 'RMSE': round(r['RMSE'], 2), 'MAE': round(r['MAE'], 2),
     'R²': round(r['R2'], 4), 'MAPE (%)': round(r['MAPE'], 2)}
    for r in [res_train, res_val, res_test]
])

print('\n📋 Resumen de métricas — Modelo LSTM')
print('='*55)
print(results_df.to_string(index=False))
print('='*55)
print(f'Parámetros totales: {model.count_params():,}')

In [ ]:
# ── Guardar modelo ────────────────────────────────────────────────────────────
os.makedirs('saved_models', exist_ok=True)
model.save('saved_models/lstm_tweet_predictor.keras')
print('✅ Modelo guardado en saved_models/lstm_tweet_predictor.keras')